# Inspect exports, then execute with PySD

The scientific specification is shared; file representations and numerical methods differ. This notebook uses both implementations and both cases without invoking Vensim or Stella Architect. Install `fair-hunter-prey[teaching]` in this kernel first (see the teaching guide). Original work: Raphael Ginster, Matthias Papesch and Vasiliy Seibert, based on Deaton and MacDonald’s *System Dynamics Learning Guide*. Model-derived material: CC BY-NC-SA 4.0.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import json
import pandas as pd
from IPython.display import display, Image
from fair_hunter_prey import inspect_results, run_example
output = Path(mkdtemp(prefix="hunter-prey-teaching-"))
print("New output directory:", output)

## 1. Read saved native exports

The original CSV readers handle separators, decimal conventions and tool-specific layouts. Historical observations have their own time axis. Unknown units remain empty. **Stella Case 2 contains final values only**, so it must not appear as a native trajectory.

In [ ]:
saved = inspect_results(output / "inspection")
trajectories = pd.read_csv(saved["trajectories"])
final_values = pd.read_csv(saved["final_values"])
display(trajectories.groupby(["implementation", "case", "variable"], sort=False).agg(samples=("time", "size"), start=("time", "min"), end=("time", "max")))
display(final_values)
display(Image(filename=str(saved["figure"])))
assert trajectories.query("implementation == 'stella' and case == 'case2'").empty
assert not json.loads(saved["provenance"].read_text())["simulation_executed"]

## 2. Compare the cases

Case 2 changes **four parameters together**. The outcome cannot be attributed to predator removal alone. Units below are model-declared; they are not newly inferred from the Stella result CSV.

In [ ]:
display(pd.DataFrame({"Parameter": ["Annual kills per predator", "Deer carrying-capacity horizon", "Desired consumption per deer", "Fraction of predators removed annually"], "Case 1": [40, 2, 0.75, 0], "Case 2": [20, 4, 0.5, 0.2], "Model unit": ["deer/(predator*year)", "year", "ton/(deer*year)", "1/year"]}))

## 3. Execute each implementation and case independently

PySD translates model copies using the repository’s unchanged compatibility adapters. Vensim uses RK2 midpoint; Stella uses RK2 Heun with held STEP evaluation. Both integrate with the model’s 0.05-year time step. Vensim saves annual samples to match the established reference check. These are **new PySD runs**, not native application runs. PySD support has feature limits; see its [documentation](https://pysd.readthedocs.io/en/latest/).

In [ ]:
runs = {}
records = []
for implementation in ("vensim", "stella"):
    for case in ("case1", "case2"):
        result = run_example(implementation, case, output / f"{implementation}-{case}")
        runs[implementation, case] = result
        record = json.loads(result["provenance"].read_text())
        records.append({key: record[key] for key in ("implementation", "case", "engine", "method", "save_interval_years", "revision")})
display(pd.DataFrame(records))

## 4. Inspect execution evidence

Compare each run with its own native reference, as done by the repository tests. The original tolerance is 0.1% for Vensim Case 1 and 5% for Case 2 on the established annual comparison grid. Stella’s reference comparison uses relative tolerance 1e-9; its Case 2 supports only a final-value comparison. Do not require identical trajectories between tools.

In [ ]:
record = json.loads(runs["stella", "case2"]["provenance"].read_text())
display(pd.DataFrame(record["inputs"].items(), columns=["Input", "SHA-256"]))
print("Actual output:", runs["stella", "case2"]["csv"])
print("Execution engine:", record["engine"])
print("All artifacts:", output)